# Choosing blocks for higher-order finite elements

Which coefficients should be corrected together when the polynomial degree increases? We keep one triangular mesh of the unit square and solve the same homogeneous Dirichlet problem in the spaces $V_h^p\subset H^1_0(\Omega)$ for $p=1,2,4$.

$$
\begin{align*}
\int_\Omega \nabla u_h\cdot\nabla v_h\,dx
&=\int_\Omega (1+x+4xy^2)\,v_h\,dx
\qquad\text{for all }v_h\in V_h^p .
\end{align*}
$$

For each degree we compare how the preconditioned conjugate gradient method (PCG) behaves when a correction acts on a single coefficient, on the coefficients of one mesh entity, or on a whole vertex patch. We also add a global correction consisting of piecewise affine functions on this same mesh. A second experiment uses a hierarchy of refined meshes to compare multigrid on the full system with multigrid and BDDC after static condensation.

In [ ]:
from ngsolve import *
from ngsolve.meshes import MakeStructured2DMesh
import matplotlib.pyplot as plt

degrees = (1, 2, 4)
mesh = MakeStructured2DMesh(quads=False, nx=16, ny=16)
source = 1 + x + 4 * x * y**2
tol = 1e-8

## Two choices of local blocks

NGSolve's hierarchical $H^1$ basis assigns one coefficient to each vertex, $p-1$ to each edge and $(p-1)(p-2)/2$ to the interior of each triangle. An **entity block** contains the free coefficients belonging to one such entity. In two dimensions, `mesh.faces` enumerates the triangles. For $p=1$ and $p=2$ every nonempty entity block is a singleton. At $p=4$ each interior edge carries three coefficients and each triangle has three interior coefficients.

In [ ]:
def entity_blocks(mesh, V):
    free = V.FreeDofs()
    blocks = []
    for entities in (mesh.vertices, mesh.edges, mesh.faces):
        for entity in entities:
            block = [dof for dof in V.GetDofNrs(entity) if dof >= 0 and free[dof]]
            if block:
                blocks.append(block)
    return blocks

A **vertex patch** collects the free coefficients of all triangles that touch one vertex. Adjacent patches overlap. Given a residual $r$, each patch solves its local system with the corresponding entries of $r$, and all local corrections are added. For a block $B$, let $R_B$ extract its coefficients and $R_B^\top$ insert them back into the full vector. The inverse action is

$$
\begin{align*}
C^{-1}r=\sum_B R_B^\top (R_B A R_B^\top)^{-1}R_B r .
\end{align*}
$$

With disjoint entity blocks this is ordinary block Jacobi. With vertex patches it is overlapping block Jacobi, also called additive Schwarz.

In [ ]:
def vertex_patches(mesh, V):
    free = V.FreeDofs()
    blocks = []
    for vertex in mesh.vertices:
        patch = set()
        for element in vertex.elements:
            for dof in V.GetDofNrs(element):
                if dof >= 0 and free[dof]:
                    patch.add(dof)
        if patch:
            blocks.append(sorted(patch))
    return blocks

## A global correction on the same mesh

A patch covers only a few triangles. The embedded space $V_h^1\subset V_h^p$ can also correct a slowly varying error across the whole domain. Let $P_1$ insert the free vertex coefficients into the fine vector. In the hierarchical basis the vertex shapes are the piecewise affine hats, so setting all higher-order coefficients to zero realizes this embedding exactly. The added inverse action is

$$
\begin{align*}
P_1 (P_1^\top A P_1)^{-1}P_1^\top r .
\end{align*}
$$

The mask below selects the corresponding principal stiffness block. Both the patch correction and this global correction use the same residual. At $p=1$ the vertex solve already solves the whole free system, so we show the combined method only for $p>1$.

In [ ]:
def p1_correction(mesh, V, A):
    vertices = BitArray(V.ndof)
    vertices[:] = False
    for vertex in mesh.vertices:
        for d in V.GetDofNrs(vertex):
            vertices[d] = True
    vertices &= V.FreeDofs()
    return A.Inverse(vertices, inverse="sparsecholesky")

## PCG with a common stopping test

All methods start from $U^{(0)}=0$. The PCG loop from Chapter 7 recomputes $\|b-AU^{(k)}\|_2/\|b\|_2$ to decide when to stop. We use the same tolerance for every inverse action.

In [ ]:
def pcg(A, b, C_inverse, tol, max_steps=200):
    U = b.CreateVector()
    U[:] = 0
    r = b.Copy()
    initial_norm = Norm(r)
    if initial_norm == 0:
        return U, [0.0]

    z = b.CreateVector()
    z.data = C_inverse * r
    d = z.Copy()
    rho = InnerProduct(r, z)
    residuals = [1.0]
    q = b.CreateVector()

    for k in range(max_steps):
        q.data = A * d
        tau = rho / InnerProduct(d, q)
        U.data += tau * d
        r.data -= tau * q

        residuals.append(Norm(b - A * U) / initial_norm)
        if residuals[-1] <= tol:
            return U, residuals

        z.data = C_inverse * r
        rho_new = InnerProduct(r, z)
        d = (z + (rho_new / rho) * d).Evaluate()
        rho = rho_new

    raise RuntimeError("PCG reached its iteration limit")

## Vary the degree

The projector $P$ keeps the free coefficients and zeroes the prescribed boundary entries. Thus the residual measured with $A=P\,\mathrm{mat}(a)\,P$ and the projected load is the residual of the free system. The entries in the table are PCG iterations.

In [ ]:
histories = {}
names = ("Point Jacobi", "Entity blocks", "Vertex patches", "Patches + P1")
print(f"{'p':>2} {'free dofs':>10}", *(f"{name:>15}" for name in names))
for p in degrees:
    V = H1(mesh, order=p, dirichlet=".*")
    u, v = V.TnT()
    a = BilinearForm(grad(u) * grad(v) * dx).Assemble()
    load = LinearForm(source * v * dx(bonus_intorder=3)).Assemble()
    free = V.FreeDofs()
    P = Projector(free, True)
    A = P @ a.mat @ P
    b = (P * load.vec).Evaluate()

    patches = a.mat.CreateBlockSmoother(vertex_patches(mesh, V))
    methods = {
        "Point Jacobi": a.mat.CreateSmoother(free),
        "Entity blocks": a.mat.CreateBlockSmoother(entity_blocks(mesh, V)),
        "Vertex patches": patches,
    }
    if p > 1:
        methods["Patches + P1"] = patches + p1_correction(mesh, V, a.mat)

    histories[p] = {}
    for name, C_inverse in methods.items():
        U, residuals = pcg(A, b, C_inverse, tol)
        histories[p][name] = residuals
    counts = [str(len(histories[p][name]) - 1) if name in methods else "—"
              for name in names]
    print(f"{p:2d} {free.NumSet():10d}", *(f"{count:>15}" for count in counts))

The residual histories below are for the largest degree. Two methods can take the same number of steps even when their intermediate residuals differ. Each patch step also solves larger local systems than an entity-block step.

In [ ]:
p = max(degrees)
fig, ax = plt.subplots(figsize=(7, 3.6))
for (name, residuals), style in zip(histories[p].items(), ("-", "--", "-.", ":")):
    ax.semilogy(residuals, linestyle=style, label=name)
ax.axhline(tol, color="0.45", linestyle=":", label="Tolerance")
ax.set(xlabel="PCG iteration", ylabel=r"$\|b-AU^{(k)}\|_2/\|b\|_2$",
       title=f"Fixed mesh, polynomial degree {p}")
ax.grid(True, which="major", alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

Further reading: The block constructions follow [iFEM's high-order Schwarz example](https://jschoeberl.github.io/iFEM/subspacecorrection/pversion.html) and the [NGSolve tutorial on local preconditioners](https://docu.ngsolve.org/latest/i-tutorials/unit-2.1.2-blockjacobi/blockjacobi.html).

## Static condensation: which equations remain global?

Chapter 7 separates the free coefficients into element interiors $L$ and the interface $E$ (vertices and edges here). The **trace** is the restriction of the finite element function to the triangle edges and is represented by $U_E$. Each interior shape function is supported on one triangle. Thus $A_{LL}^{-1}$ consists of independent element solves. Solving the interior equations for $U_L$ and substituting into the interface equations gives

$$
\begin{align*}
S U_E &= g, &
S &= A_{EE}-A_{EL}A_{LL}^{-1}A_{LE}, &
g &= b_E-A_{EL}A_{LL}^{-1}b_L.
\end{align*}
$$

$S$ is the **Schur complement**. With `condense=True`, NGSolve assembles $S$ on the interface degrees of freedom. Its transpose harmonic-extension operator supplies the correction to the load. Once PCG returns $U_E$, we recover the interior coefficients as

$$
\begin{align*}
U_L &= \underbrace{-A_{LL}^{-1}A_{LE}U_E}_{\text{harmonic extension of the trace}}
       +\underbrace{A_{LL}^{-1}b_L}_{\text{interior response to the load}} .
\end{align*}
$$

The first term fills each triangle with the discrete homogeneous solution for its prescribed trace. The second adds the effect of the element load with zero trace. Together they give the same finite element solution as the full system. On triangles $P_2$ has no interior modes, $P_4$ has three per cell, and $P_6$ has ten. This makes $p=2$ a useful control in the comparison below.

### Two ways to precondition the interface system

**High-order multigrid (MG).** In the first experiment the piecewise affine correction used an exact solve on one mesh. Here NGSolve applies a geometric V-cycle to the low-order space. It smooths, transfers the remaining residual to a coarser mesh, prolongates the coarse correction and smooths on the way back. Only the coarsest low-order problem is solved directly. A smoothing sweep solves small local correction problems of the kind introduced above. These blocks handle the higher-order coefficients on the current mesh. After condensation, the higher-order coefficients left to smooth are edge modes.

**Element-wise BDDC** stands for *Balancing Domain Decomposition by Constraints*. Think of two triangles sharing an edge. In the assembled space both triangles use the same edge coefficients. For the preconditioner we build an auxiliary space with shared vertex values (the **primal constraints** in this 2D example) and a separate copy of each higher-order edge coefficient on each incident triangle. This permits local solves for the duplicated coefficients and a global solve coupling the vertex values. Weighted averaging then combines the two edge corrections into a single conforming correction. The shared vertex constraints provide BDDC's coarse problem even on a single mesh. If $E_{\rm av}$ denotes that averaging and $\widetilde S$ the partially assembled interface operator, the inverse action is

$$
\begin{align*}
C_{\rm BDDC}^{-1} &= E_{\rm av}\,\widetilde S^{-1}E_{\rm av}^{\top}.
\end{align*}
$$

Condensation eliminates interior unknowns exactly. MG and BDDC are two choices for accelerating PCG on the remaining system. BDDC can use the same fine mesh without transfers between mesh levels.

### Compare at the same accuracy

We start with $4\times4$ squares split into triangles and refine twice, keeping $p$ fixed within each hierarchy. At every level all three methods use the same mesh, space, load and zero initial guess for the global unknowns. The two condensed runs share the very same Schur complement. We register the preconditioners before assembly so MG sees the refinement history and BDDC has access to the element matrices.

For a recovered vector $U=(U_L,U_E)$, the original interior residual is zero up to roundoff, and its interface residual is $g-SU_E$. We therefore stop all methods at $\|b-AU\|_2\leq\mathrm{tol}\,\|b\|_2$ on the original free space. The PCG helper divides by the norm of its own right-hand side, so the condensed runs pass the relative tolerance $\mathrm{tol}\,\|b\|_2/\|g\|_2$. We check the full residual explicitly after recovery.

In [ ]:
def compare_on_hierarchy(p, coarse_n=4, levels=3):
    grid = MakeStructured2DMesh(quads=False, nx=coarse_n, ny=coarse_n)
    space = H1(grid, order=p, dirichlet=".*", autoupdate=True)
    u, v = space.TnT()
    stiffness = grad(u) * grad(v) * dx
    full = BilinearForm(stiffness)
    condensed = BilinearForm(stiffness, condense=True)
    load = LinearForm(source * v * dx(bonus_intorder=3))

    mg_full = Preconditioner(full, "multigrid", cycle=1, inverse="sparsecholesky")
    mg_trace = Preconditioner(condensed, "multigrid", cycle=1, inverse="sparsecholesky")
    bddc_trace = Preconditioner(condensed, "bddc", inverse="sparsecholesky")

    for level in range(levels):
        if level:
            grid.Refine()
        full.Assemble()
        condensed.Assemble()
        load.Assemble()

        free = space.FreeDofs()
        interface = space.FreeDofs(coupling=True)
        P = Projector(free, True)
        Q = Projector(interface, True)
        A = P @ full.mat @ P
        S = Q @ condensed.mat @ Q
        b = (P * load.vec).Evaluate()
        g = (Q * (load.vec + condensed.harmonic_extension_trans * load.vec)).Evaluate()

        U, history = pcg(A, b, mg_full.mat, tol)
        counts = [len(history) - 1]
        residuals = [Norm(b - A * U) / Norm(b)]
        for pre in (mg_trace, bddc_trace):
            trace, history = pcg(S, g, pre.mat, tol * Norm(b) / Norm(g))
            U = (trace + condensed.harmonic_extension * trace
                 + condensed.inner_solve * load.vec).Evaluate()
            counts.append(len(history) - 1)
            residuals.append(Norm(b - A * U) / Norm(b))

        yield level, free.NumSet(), interface.NumSet(), counts, max(residuals)

In [ ]:
print(f"{'p':>2} {'level':>5} {'full dofs':>10} {'trace dofs':>11} "
      f"{'MG full':>9} {'MG trace':>10} {'BDDC trace':>12} {'max rel. residual':>18}")
for p in (2, 4, 6):
    for level, n_free, n_trace, counts, residual in compare_on_hierarchy(p):
        print(f"{p:2d} {level:5d} {n_free:10d} {n_trace:11d} "
              f"{counts[0]:9d} {counts[1]:10d} {counts[2]:12d} {residual:18.2e}")

**Read the table.** The three method columns count PCG steps. On the finest default mesh at $p=6$, condensation reduces the global system from 9025 to 3905 unknowns. MG takes 21 steps on the full system and 7 on the trace system. Element-wise BDDC takes 31 steps on that same trace system. First compare the two MG columns at $p=2$, then at $p=4$ and $p=6$. How does the gap relate to the number of eliminated unknowns? Next compare MG and BDDC across the three levels at fixed $p$, using the two trace columns. They solve identical global equations with different inverse actions. A comparison of running times would also include setup, work per iteration and interior recovery.

Further reading: [NGSolve static condensation](https://docu.ngsolve.org/latest/i-tutorials/unit-1.4-staticcond/staticcond.html) · [High-order multigrid experiments](https://docu.ngsolve.org/latest/i-tutorials/unit-2.1.1-preconditioners/preconditioner.html) · [Element-wise BDDC](https://docu.ngsolve.org/latest/i-tutorials/unit-2.1.4-bddc/bddc.html)